# Fixture context — home vs away (`total_points`)

*Read-only informative artifact. This notebook characterises how home advantage
changes the target picture so a human can decide how to read it. It produces no
gate decisions and no PROCEED/STOP verdict.*

## Questions a manager asks about home advantage

- **How much is home advantage worth in points?** Everyone "knows" home is
  better — but is it a haul's worth, or a rounding error?
- **Does it differ by position?** Do attackers feed off a home crowd more than
  defenders, whose returns hinge on clean sheets that home advantage also helps?
- **Does the home edge show up as fewer blanks, more returns, or both?**

Everything below is **season-pooled** over the study range and stratified by
venue.

## Setup — and the SGW restriction

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), then split by venue
using `was_home`.

**Critical restriction: SGW rows only.** `was_home` is clean for single
gameweeks but **ambiguous for double gameweeks** — a DGW player can play one
fixture home and one away, yet the row carries a single `was_home` flag and a
doubled `total_points`. Including DGW rows would mislabel venue and contaminate
the comparison with fixture-doubling. So this notebook is **restricted to
`fixture_context == "SGW"`**, where `was_home` unambiguously labels the single
fixture. **DGW rows are excluded from this analysis for that reason** (their
double-vs-single treatment lives in `dgw_vs_sgw.ipynb`).

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. The 60-minute performance-boundary question is **deferred
to the `population/` layer** and is deliberately not baked in here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. The 60-minute
# performance boundary is NOT imposed here -- deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

# CRITICAL: restrict to SGW so `was_home` is unambiguous. DGW rows can be one
# home + one away under a single was_home flag and a doubled total -> excluded.
df = df[df["fixture_context"] == "SGW"].copy()
df = df[df["was_home"].notna()].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]
VENUES = ["home", "away"]
df["venue"] = np.where(df["was_home"], "home", "away")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print("Venue analysis RESTRICTED to SGW rows (was_home unambiguous); DGW rows excluded.")
print(f"Population: minutes > 0, SGW only, n = {len(df):,} player-gameweeks")
print("Venue counts:")
for v in VENUES:
    print(f"  {v:<5}: {int((df['venue'] == v).sum()):>6,}")

## (a) `total_points` distribution home vs away by position

**What we measure** — the distribution of `total_points` (count, mean, median,
spread, p90) for home and away player-gameweeks, **within each position**, via
`compare_cohorts` with cohorts = {home, away}. SGW rows only.

*Stat glossary for this table:* **mean** — arithmetic average, pulled up by rare hauls; **median** — middle value, the robust "typical" return; **std** — typical distance from the mean in points, sensitive to outliers; **IQR (p75−p25)** — spread of the middle 50%, robust to skew; **p90/p99** — score exceeded by only 10%/1% of appearances (practical ceiling); **skew** — positive = long right tail of rare high scores.

**What it means** — this is the direct read on **how much home advantage is
worth in points**: the home-minus-away gap in mean/median `total_points`,
position by position, is what a manager gains (in expectation) from a home
fixture. It informs venue-aware captaincy and transfer timing — e.g. whether a
home fixture is worth captaining a defender on, not just an attacker.

**What it doesn't mean** — this is **association, descriptively shown**, not a
causal or predictive claim, and it is **season-pooled** across all players and
weeks. The home/away split is roughly balanced by construction (every team
plays home and away), so team-quality confounding is milder here than for FDR —
but venue still correlates with opponent strength in any given week. It is not a
forecast for a specific player-week.

**Guiding question** — *How much is home advantage worth in points, and does it
differ by position?*

In [ ]:
va_rows = []
for pos in POSITIONS:
    sub = df[df.position == pos]
    cohorts = {v: sub[sub["venue"] == v] for v in VENUES}
    stats = compare_cohorts(cohorts, value_col="total_points")
    for v in VENUES:
        s = stats.loc[v]
        va_rows.append({
            "position": pos,
            "venue": v,
            "n": int(s["count"]) if not np.isnan(s["count"]) else 0,
            "mean": s["mean"],
            "median": s["median"],
            "std": s["std"],
            "p90": s["p90"],
        })
venue_by_pos = pd.DataFrame(va_rows)
display(venue_by_pos.round(2))

# Home-minus-away mean gap per position -- the headline home-advantage number.
gap_rows = []
for pos in POSITIONS:
    p = venue_by_pos[venue_by_pos.position == pos].set_index("venue")
    gap_rows.append({
        "position": pos,
        "mean_home": p.loc["home", "mean"],
        "mean_away": p.loc["away", "mean"],
        "home_minus_away": round(p.loc["home", "mean"] - p.loc["away", "mean"], 2),
    })
venue_gap = pd.DataFrame(gap_rows)
display(venue_gap.round(2))

## (b) Blank rate and return rate home vs away by position

**What we measure** — within each (position, venue) cell, the share of
player-gameweeks that **blank** (`total_points == 0`) and the share that
**return** (`total_points >= 4`). Computed inline as simple proportions. SGW
rows only.

**What it means** — this decomposes the home-minus-away gap from (a) into
*mechanism*: does playing at home help by **avoiding blanks** (floor), by
**producing more returns** (ceiling), or both? For defenders and keepers the
channel is likely clean-sheet-driven (fewer blanks at home); for attackers it
may be more about extra returns.

**What it doesn't mean** — these are **season-pooled** rates over the
**participation** population, so a blank includes brief cameos that returned
nothing (no 60-minute gate — deferred to `population/`). Venue still correlates
with weekly opponent strength, so the rates are descriptive association, not
predictive probabilities for a specific player-week.

**Guiding question** — *Does the home edge show up as fewer blanks, more
returns, or both — and does the channel differ by position?*

In [ ]:
venue_rates = (
    df.groupby(["position", "venue"])["total_points"]
    .apply(lambda y: pd.Series({
        "n": len(y),
        "blank_rate_%": round((y == 0).mean() * 100, 1) if len(y) else float("nan"),
        "return_4+_%": round((y >= 4).mean() * 100, 1) if len(y) else float("nan"),
    }))
    .reset_index()
)
display(venue_rates)


## What home advantage looks like

Plain-language summary of the descriptive picture (not a verdict):

- The venue table and the **home-minus-away mean gap** (a) size how much, in
  `total_points`, playing at home is worth — position by position.
- The blank/return table (b) shows the *channel*: whether home helps mainly by
  cutting blanks (floor) or lifting 4+ returns (ceiling), and whether that
  differs across positions.
- Any gap is **descriptive association**, not causation, and the home/away
  split is roughly balanced so team-quality confounding is milder than for FDR
  — though weekly opponent strength still correlates with venue.

All figures are **whole-season**, season-pooled, over the **participation**
population (`minutes > 0` — not a performance gate; the 60-minute boundary is
deferred to the `population/` layer), and **restricted to SGW rows** so
`was_home` is unambiguous. **DGW rows are excluded** from this venue analysis
because their `was_home` flag cannot label a one-home-one-away round — their
treatment lives in `dgw_vs_sgw.ipynb`. Fixture difficulty is treated in
`fdr_context.ipynb`.